# Project 6 — Advanced Push Pipeline with Generator Coroutines

This notebook implements a reusable **push-based CSV pipeline**:

```text
CSV source → normalization → validation → predicate filter → CSV sink
```

Added-value features:

- automatic delimiter detection for comma, semicolon, tab, and pipe files;
- optional automatic header detection;
- explicit delimiter overrides when detection is undesirable;
- composable `all`, `any`, and negated predicates;
- configurable case-sensitive or case-insensitive matching;
- row normalization and schema validation;
- detailed processing statistics;
- configurable output delimiter and optional output headers;
- self-contained smoke tests for both semicolon and comma CSV files;
- Python 3.6-compatible implementation.

## 1. Imports and coroutine helper

Generator-based coroutines must be advanced to their first `yield` before
receiving values through `.send(...)`. The decorator handles that setup.

In [1]:
import csv
from functools import wraps
from pathlib import Path


def coroutine(generator_function):
    """Create and automatically prime a generator-based coroutine."""
    @wraps(generator_function)
    def start(*args, **kwargs):
        generator = generator_function(*args, **kwargs)
        next(generator)
        return generator

    return start

## 2. CSV format detection

`csv.Sniffer` is used first. If it cannot infer the dialect, a deterministic
fallback chooses the delimiter that appears most consistently across the
sample's non-empty lines.

The source file is rewound after sampling, so no data is lost.

In [2]:
def _fallback_delimiter(sample, delimiters):
    """Choose the most consistently occurring delimiter in *sample*."""
    lines = [line for line in sample.splitlines() if line.strip()][:20]

    if not lines:
        return ","

    def score(delimiter):
        counts = [line.count(delimiter) for line in lines]
        positive_counts = [count for count in counts if count > 0]

        if not positive_counts:
            return (-1, -1, -1)

        consistency = -(
            max(positive_counts) - min(positive_counts)
        )
        coverage = len(positive_counts)
        frequency = sum(positive_counts)
        return (coverage, consistency, frequency)

    return max(delimiters, key=score)


def _dialect_for_delimiter(delimiter):
    """Return a standard CSV dialect configured for one delimiter."""
    class DetectedDialect(csv.excel):
        pass

    DetectedDialect.delimiter = delimiter
    return DetectedDialect


def sniff_csv_format(
    file_object,
    explicit_delimiter=None,
    detect_header=True,
    delimiters=",;\t|",
    sample_size=16384,
):
    """Inspect a text file and return its dialect and header information."""
    if explicit_delimiter is not None:
        if not isinstance(explicit_delimiter, str) or len(explicit_delimiter) != 1:
            raise ValueError(
                "explicit_delimiter must be one character or None"
            )

    sample = file_object.read(sample_size)
    file_object.seek(0)

    if explicit_delimiter is not None:
        dialect = _dialect_for_delimiter(explicit_delimiter)
        delimiter_source = "explicit"
    elif not sample:
        dialect = csv.excel
        delimiter_source = "default"
    else:
        try:
            dialect = csv.Sniffer().sniff(sample, delimiters=delimiters)
            delimiter_source = "sniffer"
        except csv.Error:
            delimiter = _fallback_delimiter(sample, delimiters)
            dialect = _dialect_for_delimiter(delimiter)
            delimiter_source = "fallback"

    header_detected = False
    if detect_header and sample:
        try:
            header_detected = csv.Sniffer().has_header(sample)
        except csv.Error:
            header_detected = False

    return {
        "dialect": dialect,
        "delimiter": dialect.delimiter,
        "quotechar": dialect.quotechar,
        "header_detected": header_detected,
        "delimiter_source": delimiter_source,
    }

## 3. Predicate composition

Filtering is separated from pipeline mechanics. Small predicates can be reused
and composed with `all_of`, `any_of`, and `negate`.

In [3]:
def contains_text(fragment, field_index=0, case_sensitive=False):
    """Return a predicate checking whether text occurs in one field."""
    if not isinstance(fragment, str):
        raise TypeError("fragment must be a string")
    if fragment == "":
        raise ValueError("fragment must not be empty")
    if field_index < 0:
        raise ValueError("field_index must be non-negative")

    needle = fragment if case_sensitive else fragment.casefold()

    def predicate(row):
        if field_index >= len(row):
            return False

        value = row[field_index]
        haystack = value if case_sensitive else value.casefold()
        return needle in haystack

    return predicate


def all_of(*predicates):
    """Return a predicate requiring every supplied predicate to pass."""
    return lambda row: all(predicate(row) for predicate in predicates)


def any_of(*predicates):
    """Return a predicate requiring at least one predicate to pass."""
    return lambda row: any(predicate(row) for predicate in predicates)


def negate(predicate):
    """Return the logical inverse of a predicate."""
    return lambda row: not predicate(row)


def build_fragment_predicate(
    fragments,
    field_index=0,
    match_mode="all",
    case_sensitive=False,
):
    """Create one predicate from any number of text fragments.

    `match_mode="all"` requires every fragment.
    `match_mode="any"` requires at least one fragment.
    An empty fragment sequence matches every row.
    """
    fragments = tuple(fragments)

    if match_mode not in ("all", "any"):
        raise ValueError("match_mode must be 'all' or 'any'")

    if not fragments:
        return lambda row: True

    predicates = tuple(
        contains_text(
            fragment,
            field_index=field_index,
            case_sensitive=case_sensitive,
        )
        for fragment in fragments
    )

    return all_of(*predicates) if match_mode == "all" else any_of(*predicates)

## 4. Statistics and coroutine stages

The statistics dictionary is shared by the stages. This keeps each coroutine
small while producing a useful run report.

In [4]:
def new_pipeline_stats():
    """Return a fresh statistics dictionary for one pipeline run."""
    return {
        "rows_read": 0,
        "rows_evaluated": 0,
        "rows_matched": 0,
        "rows_written": 0,
        "rows_skipped_blank": 0,
        "rows_skipped_invalid": 0,
        "header_skipped": False,
        "input_header": None,
        "input_delimiter": None,
        "delimiter_source": None,
        "header_detected": False,
    }


@coroutine
def csv_sink(
    output_path,
    stats,
    delimiter=",",
    include_header=False,
    header=None,
    encoding="utf-8",
):
    """Receive row sequences and write them to a CSV file."""
    output_path = Path(output_path)

    if not isinstance(delimiter, str) or len(delimiter) != 1:
        raise ValueError("output delimiter must be exactly one character")
    if include_header and header is None:
        raise ValueError("header must be provided when include_header=True")

    output_path.parent.mkdir(parents=True, exist_ok=True)

    with output_path.open("w", newline="", encoding=encoding) as output_file:
        writer = csv.writer(
            output_file,
            delimiter=delimiter,
            lineterminator="\n",
        )

        if include_header:
            writer.writerow(header)

        try:
            while True:
                row = yield
                writer.writerow(row)
                stats["rows_written"] += 1
        except GeneratorExit:
            return


@coroutine
def filter_rows(predicate, target, stats):
    """Forward rows satisfying *predicate* to the downstream stage."""
    if not callable(predicate):
        raise TypeError("predicate must be callable")

    try:
        while True:
            row = yield
            stats["rows_evaluated"] += 1

            if predicate(row):
                stats["rows_matched"] += 1
                target.send(row)
    except GeneratorExit:
        target.close()
        return

## 5. CSV source

The source performs format detection, optional header handling, normalization,
blank-row removal, and column-count validation before pushing rows downstream.

`skip_header` accepts `True`, `False`, or `"auto"`.

In [5]:
def _normalize_row(row, strip_fields=True):
    """Normalize one parsed CSV row."""
    if strip_fields:
        row = [field.strip() for field in row]

    # Handle a UTF-8 BOM even when a caller chooses plain utf-8 encoding.
    if row:
        row[0] = row[0].lstrip("\ufeff")

    return row


def push_csv(
    source_path,
    target,
    stats,
    input_delimiter=None,
    skip_header="auto",
    detect_header=True,
    strip_fields=True,
    skip_blank_rows=True,
    expected_columns=None,
    invalid_row_policy="raise",
    encoding="utf-8-sig",
):
    """Read a CSV file and push validated rows into *target*.

    `invalid_row_policy` may be `"raise"` or `"skip"`.
    """
    source_path = Path(source_path)

    if skip_header not in (True, False, "auto"):
        raise ValueError("skip_header must be True, False, or 'auto'")
    if invalid_row_policy not in ("raise", "skip"):
        raise ValueError("invalid_row_policy must be 'raise' or 'skip'")
    if expected_columns is not None and expected_columns <= 0:
        raise ValueError("expected_columns must be positive or None")

    try:
        with source_path.open("r", newline="", encoding=encoding) as source_file:
            format_info = sniff_csv_format(
                source_file,
                explicit_delimiter=input_delimiter,
                detect_header=detect_header,
            )

            stats["input_delimiter"] = format_info["delimiter"]
            stats["delimiter_source"] = format_info["delimiter_source"]
            stats["header_detected"] = format_info["header_detected"]

            reader = csv.reader(
                source_file,
                dialect=format_info["dialect"],
                skipinitialspace=True,
            )

            should_skip_header = (
                format_info["header_detected"]
                if skip_header == "auto"
                else skip_header
            )

            if should_skip_header:
                header = next(reader, None)
                if header is not None:
                    stats["input_header"] = _normalize_row(
                        header,
                        strip_fields=strip_fields,
                    )
                    stats["header_skipped"] = True

            for row_number, row in enumerate(reader, start=2 if should_skip_header else 1):
                row = _normalize_row(row, strip_fields=strip_fields)

                if skip_blank_rows and not any(row):
                    stats["rows_skipped_blank"] += 1
                    continue

                stats["rows_read"] += 1

                if expected_columns is not None and len(row) != expected_columns:
                    stats["rows_skipped_invalid"] += 1
                    message = (
                        "Row {} has {} columns; expected {}. Row: {!r}".format(
                            row_number,
                            len(row),
                            expected_columns,
                            row,
                        )
                    )

                    if invalid_row_policy == "raise":
                        raise ValueError(message)

                    continue

                target.send(row)
    finally:
        target.close()

    return stats

## 6. Pipeline builder and high-level runner

The public runner validates paths, builds the coroutine chain, executes the
source, and returns a detailed statistics dictionary.

In [6]:
def build_name_filter_pipeline(
    output_path,
    fragments,
    stats,
    name_field_index=0,
    match_mode="all",
    case_sensitive=False,
    output_delimiter=",",
    include_header=False,
    output_header=None,
    encoding="utf-8",
):
    """Build the filter and sink stages for a name-based pipeline."""
    predicate = build_fragment_predicate(
        fragments=fragments,
        field_index=name_field_index,
        match_mode=match_mode,
        case_sensitive=case_sensitive,
    )

    sink = csv_sink(
        output_path=output_path,
        stats=stats,
        delimiter=output_delimiter,
        include_header=include_header,
        header=output_header,
        encoding=encoding,
    )

    return filter_rows(predicate, sink, stats)


def run_name_filter_pipeline(
    source_path,
    output_path,
    fragments,
    name_field_index=0,
    match_mode="all",
    case_sensitive=False,
    input_delimiter=None,
    output_delimiter=",",
    skip_header="auto",
    detect_header=True,
    strip_fields=True,
    expected_columns=None,
    invalid_row_policy="raise",
    include_header=False,
    output_header=None,
    input_encoding="utf-8-sig",
    output_encoding="utf-8",
):
    """Build and execute the complete CSV name-filtering pipeline."""
    source_path = Path(source_path)
    output_path = Path(output_path)
    fragments = tuple(fragments)

    if not source_path.is_file():
        raise FileNotFoundError("Source CSV not found: {}".format(source_path))
    if source_path.resolve() == output_path.resolve():
        raise ValueError("source_path and output_path must be different files")

    stats = new_pipeline_stats()
    stats.update({
        "source": str(source_path),
        "output": str(output_path),
        "filters": fragments,
        "match_mode": match_mode,
    })

    pipeline = build_name_filter_pipeline(
        output_path=output_path,
        fragments=fragments,
        stats=stats,
        name_field_index=name_field_index,
        match_mode=match_mode,
        case_sensitive=case_sensitive,
        output_delimiter=output_delimiter,
        include_header=include_header,
        output_header=output_header,
        encoding=output_encoding,
    )

    return push_csv(
        source_path=source_path,
        target=pipeline,
        stats=stats,
        input_delimiter=input_delimiter,
        skip_header=skip_header,
        detect_header=detect_header,
        strip_fields=strip_fields,
        expected_columns=expected_columns,
        invalid_row_policy=invalid_row_policy,
        encoding=input_encoding,
    )

## 7. Reporting and preview helpers

In [7]:
def print_run_report(stats):
    """Print a compact human-readable processing report."""
    print("CSV pipeline report")
    print("-------------------")
    print("Input delimiter : {!r} ({})".format(
        stats.get("input_delimiter"),
        stats.get("delimiter_source"),
    ))
    print("Header detected : {}".format(stats.get("header_detected")))
    print("Header skipped  : {}".format(stats.get("header_skipped")))
    print("Rows read       : {}".format(stats.get("rows_read")))
    print("Rows evaluated  : {}".format(stats.get("rows_evaluated")))
    print("Rows matched    : {}".format(stats.get("rows_matched")))
    print("Rows written    : {}".format(stats.get("rows_written")))
    print("Blank skipped   : {}".format(stats.get("rows_skipped_blank")))
    print("Invalid skipped : {}".format(stats.get("rows_skipped_invalid")))


def preview_csv(path, delimiter=",", limit=5, encoding="utf-8"):
    """Return up to *limit* parsed rows from a CSV file."""
    path = Path(path)
    rows = []

    with path.open("r", newline="", encoding=encoding) as file_object:
        reader = csv.reader(file_object, delimiter=delimiter)
        for row_number, row in enumerate(reader):
            if row_number >= limit:
                break
            rows.append(row)

    return rows

## 8. Project example

The input delimiter is left as `None`, so the sniffer detects the semicolon in
the provided `cars.csv`. The output remains a standard comma-separated file.

`expected_columns=9` catches delimiter or schema problems early.

In [8]:
SOURCE_FILE = Path("cars.csv")
OUTPUT_FILE = Path("chevrolet_monte_carlo_landau.csv")

stats = run_name_filter_pipeline(
    source_path=SOURCE_FILE,
    output_path=OUTPUT_FILE,
    fragments=("Chevrolet", "Carlo", "Landau"),
    match_mode="all",
    input_delimiter=None,
    output_delimiter=",",
    skip_header="auto",
    expected_columns=9,
    invalid_row_policy="raise",
)

print_run_report(stats)
preview_csv(OUTPUT_FILE)

CSV pipeline report
-------------------
Input delimiter : ';' (sniffer)
Header detected : True
Header skipped  : True
Rows read       : 406
Rows evaluated  : 406
Rows matched    : 2
Rows written    : 2
Blank skipped   : 0
Invalid skipped : 0


[['Chevrolet Monte Carlo Landau',
  '15.5',
  '8',
  '350.0',
  '170.0',
  '4165.',
  '11.4',
  '77',
  'US'],
 ['Chevrolet Monte Carlo Landau',
  '19.2',
  '8',
  '305.0',
  '145.0',
  '3425.',
  '13.2',
  '78',
  'US']]

## 9. Verify the required project output

In [9]:
expected_rows = [
    [
        "Chevrolet Monte Carlo Landau",
        "15.5", "8", "350.0", "170.0", "4165.", "11.4", "77", "US",
    ],
    [
        "Chevrolet Monte Carlo Landau",
        "19.2", "8", "305.0", "145.0", "3425.", "13.2", "78", "US",
    ],
]

actual_rows = preview_csv(
    OUTPUT_FILE,
    delimiter=",",
    limit=10,
)

assert actual_rows == expected_rows, (
    "Output did not match the required rows.\n"
    "Expected: {!r}\n"
    "Actual:   {!r}".format(expected_rows, actual_rows)
)

print(OUTPUT_FILE.read_text(encoding="utf-8"))

Chevrolet Monte Carlo Landau,15.5,8,350.0,170.0,4165.,11.4,77,US
Chevrolet Monte Carlo Landau,19.2,8,305.0,145.0,3425.,13.2,78,US



## 10. Additional examples

### Match any fragment

```python
run_name_filter_pipeline(
    "cars.csv",
    "ford_or_chevrolet.csv",
    fragments=("Ford", "Chevrolet"),
    match_mode="any",
    expected_columns=9,
)
```

### Copy all valid rows

```python
run_name_filter_pipeline(
    "cars.csv",
    "all_cars.csv",
    fragments=(),
    expected_columns=9,
)
```

### Skip malformed rows instead of raising

```python
run_name_filter_pipeline(
    "cars.csv",
    "valid_cars.csv",
    fragments=(),
    expected_columns=9,
    invalid_row_policy="skip",
)
```

## 11. Built-in smoke tests

These tests verify delimiter sniffing, header handling, `all` matching, `any`
matching, and comma-separated output without depending on the project file.

In [10]:
def run_smoke_tests():
    from tempfile import TemporaryDirectory

    with TemporaryDirectory() as directory:
        directory = Path(directory)

        semicolon_source = directory / "cars_semicolon.csv"
        semicolon_output = directory / "landau.csv"
        semicolon_source.write_text(
            "Chevrolet Monte Carlo Landau;15.5;8;350.0;170.0;4165.;11.4;77;US\n"
            "Chevrolet Monte Carlo Landau;19.2;8;305.0;145.0;3425.;13.2;78;US\n"
            "Ford Mustang;18.0;8;302.0;130.0;3169.;12.0;78;US\n",
            encoding="utf-8",
        )

        semicolon_stats = run_name_filter_pipeline(
            semicolon_source,
            semicolon_output,
            fragments=("Chevrolet", "Carlo", "Landau"),
            expected_columns=9,
            skip_header=False,
        )

        semicolon_rows = preview_csv(semicolon_output)
        assert semicolon_stats["input_delimiter"] == ";"
        assert semicolon_stats["rows_written"] == 2
        assert len(semicolon_rows) == 2
        assert all(len(row) == 9 for row in semicolon_rows)

        comma_source = directory / "cars_header.csv"
        comma_output = directory / "brands.csv"
        comma_source.write_text(
            "name,mpg,cylinders\n"
            "Ford Mustang,18.0,8\n"
            "Chevrolet Nova,22.0,6\n"
            "Toyota Corolla,31.0,4\n",
            encoding="utf-8",
        )

        comma_stats = run_name_filter_pipeline(
            comma_source,
            comma_output,
            fragments=("Ford", "Chevrolet"),
            match_mode="any",
            expected_columns=3,
            skip_header="auto",
        )

        comma_rows = preview_csv(comma_output)
        assert comma_stats["input_delimiter"] == ","
        assert comma_stats["header_skipped"] is True
        assert [row[0] for row in comma_rows] == [
            "Ford Mustang",
            "Chevrolet Nova",
        ]

    return "All smoke tests passed."


run_smoke_tests()

'All smoke tests passed.'